# Project 2 - Machine Model Training

Created with Grok Build


Mark Khusid

Pandas, NumPy, and pickle load the saved classifier and the test matrix. Simpson's rule is imported for the area-under-curve feature. The scikit-learn imports match the training notebook; this notebook only uses the loaded support-vector model to predict.


In [1]:
import pandas as pd
import numpy as np

import pickle

from scipy.integrate import simpson

from sklearn.impute import KNNImputer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import cross_val_score

## Load Model from Pickle File

Opens `svm_model.pkl` from the notebook directory and restores the fitted support-vector classifier into `svm_model`.


In [14]:
with open('svm_model.pkl', 'rb') as f:
    svm_model = pickle.load(f)

## Load Test Data into Matrix

Reads `test.csv` with no header row. Each row is one glucose window and each column is one sample in that window.


In [15]:
df_test = pd.read_csv('test.csv', header=None)

Copies the test table into a NumPy array named `test_matrix`. The display of that array is left commented out.


In [16]:
test_matrix = df_test.values
#test_matrix

## Feature Extraction

### Define Feature Extractor Helper Functions

The first derivative is the change in glucose divided by the 5-minute sample spacing. `np.diff` shortens the window by one sample.


In [17]:
def compute_first_derivative(glucose_data, delta_t=5):
    return np.diff(glucose_data) / delta_t

The second derivative is the discrete curvature of the glucose window, `(g[i+1] - 2 g[i] + g[i-1]) / 25`, again using a 5-minute spacing. The result is two samples shorter than the window.


In [18]:
def compute_second_derivative(glucose_values, delta_t=5):
    second_derivative = np.zeros(len(glucose_values) - 2)
    for i in range(1, len(glucose_values) - 1):
        second_derivative[i - 1] = \
            (glucose_values[i + 1] - 2 * glucose_values[i] + glucose_values[i - 1]) / (delta_t ** 2)
    return second_derivative

Left commented. The call uses the name `first_derivative`, which is not defined in this notebook; the helper above is `compute_first_derivative`.


In [19]:
#first_derivative(cleaned_imputed_meal_matrix[0])

Left commented. The call uses the name `second_derivative`, which is not defined here; the helper above is `compute_second_derivative`.


In [20]:
#second_derivative(cleaned_imputed_meal_matrix[0])

`compute_AUC` integrates one glucose window with Simpson's rule and a spacing of 5 minutes, so the result has units of mg/dL times minutes.


In [21]:
def compute_AUC(glucose_values, delta_t=5):
    return simpson(glucose_values, dx=delta_t)

### Define Feature Extractor Function

Walks each test row and builds eight numbers: mean, standard deviation, maximum, minimum, mean first derivative, mean second derivative, area under the curve, and the sample index of the peak. Windows of two samples or fewer get a second-derivative value of 0. The return value is one row of features per window.


In [22]:
def feature_extractor(data_matrix):

    features = []

    for row in data_matrix:
        mean_glucose = np.mean(row)
        std_glucose  = np.std(row)
        max_glucose  = np.max(row)
        min_glucose  = np.min(row)
        
        first_derivative = compute_first_derivative(row)
        mean_first_derivate = np.mean(first_derivative)

        if len(row) > 2:
            second_derivative = compute_second_derivative(row)
            mean_second_derivative = np.mean(second_derivative)
        else:
            mean_second_derivative = 0

        AUC = compute_AUC(row)

        time_to_peak = np.argmax(row)

        features_vector = [
            mean_glucose,
            std_glucose,
            max_glucose,
            min_glucose,
            mean_first_derivate,
            mean_second_derivative,
            AUC,
            time_to_peak
        ]

        features.append(features_vector)
    
    return np.array(features)

### Get Feature Matrix for Test Data

Runs `feature_extractor` on `test_matrix` and stores the eight-column feature array in `test_features_matrix`.


In [23]:
test_features_matrix = \
    feature_extractor(test_matrix)

## Apply Model to Test Data

Applies the loaded classifier to the test features. Each entry of `y_pred` is the predicted class for one window.


In [24]:
y_pred = svm_model.predict(test_features_matrix)

Displays `y_pred` so the class labels can be checked before they are written out.


In [25]:
y_pred

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

## Save Predictions to CSV File

Writes one integer prediction per line to `Result.csv`, with no header.


In [27]:
np.savetxt('Result.csv', y_pred.reshape(-1, 1), delimiter=',', fmt='%d')